# EduFeed ZenML Pipeline — DeBERTa + Groq LLM
**Dataset:** `balanced_edufeed_dataset.xlsx` (35 421 professor reviews)  
**Model:** `cross-encoder/nli-deberta-v3-base` for zero-shot sentiment  
**LLM:** Groq (`llama3-8b-8192`) via `https://api.groq.com/openai/v1`  
**Pipeline:** ZenML 6-layer pipeline (Ingest → Anonymise → Embed → FAISS → Groq Insights → Export)

## ① Install all dependencies

In [ ]:
import subprocess, sys

pkgs = [
    # Core ML
    "torch",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "faiss-cpu",
    "datasets",
    "scikit-learn",
    # Data
    "openpyxl",
    "pandas",
    "numpy",
    # ZenML
    "zenml",
    "sqlalchemy",
    "passlib==1.7.4",
    "bcrypt==4.0.1",
    "pymysql",
    # Groq / OpenAI-compatible client
    "openai",        # Groq uses the OpenAI Python SDK
    "groq",          # Official Groq SDK (optional but helpful)
    "requests",
]

print("Installing dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
else:
    print("All packages installed ✔")

## ② Environment — set your Groq API key

> Get a free key from https://console.groq.com → API Keys

In [ ]:
import os

# ─── SET YOUR GROQ API KEY HERE ──────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "gsk_YOUR_GROQ_API_KEY_HERE"   # <-- replace this
# ─────────────────────────────────────────────────────────────────────────────

GROQ_BASE_URL  = "https://api.groq.com/openai/v1"
GROQ_MODEL     = "llama3-8b-8192"   # fast, free-tier friendly
DATASET_PATH   = "balanced_edufeed_dataset.xlsx"   # put this xlsx next to the notebook
WORK_DIR       = os.path.abspath("edufeed_outputs")
MAX_PROFS      = 20    # professor insight cards to generate

os.makedirs(WORK_DIR, exist_ok=True)
print("WORK_DIR  :", WORK_DIR)
print("Groq model:", GROQ_MODEL)
print("Groq base :", GROQ_BASE_URL)

## ③ Initialise ZenML (local SQLite store)

In [ ]:
import shutil
from pathlib import Path

ZEN_DIR = Path(WORK_DIR) / ".zen"
if ZEN_DIR.exists():
    shutil.rmtree(ZEN_DIR)
ZEN_DIR.mkdir(parents=True, exist_ok=True)

os.environ["ZENML_CONFIG_PATH"]           = str(ZEN_DIR)
os.environ["ZENML_ANALYTICS_OPT_IN"]      = "false"
os.environ["ZENML_DEFAULT_USER_PASSWORD"]  = "admin123"

try:
    from zenml.config.global_config import GlobalConfiguration
    from zenml.zen_stores.sql_zen_store import SqlZenStoreConfiguration

    store_cfg = SqlZenStoreConfiguration(
        type="sql",
        url=f"sqlite:///{ZEN_DIR}/zenml.db"
    )
    gc = GlobalConfiguration()
    gc._configure_store(store_cfg)
    print("ZenML initialised ✔  (active stack:", gc.active_stack_id, ")")
except Exception as e:
    print("ZenML init warning (non-fatal):", e)
    print("Pipeline will still run in standalone mode.")

## ④ Groq LLM helper
Uses the official OpenAI SDK pointed at the Groq endpoint.

In [ ]:
from openai import OpenAI

def groq_chat(prompt: str, system: str = "You are a helpful assistant.",
              max_tokens: int = 512, temperature: float = 0.3) -> str:
    """
    Call the Groq LLM via the OpenAI-compatible API.
    
    Parameters
    ----------
    prompt      : user prompt
    system      : system instruction
    max_tokens  : max response tokens
    temperature : sampling temperature
    
    Returns
    -------
    str : model response text
    """
    client = OpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url=GROQ_BASE_URL,
    )
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()


# ── quick smoke-test ──────────────────────────────────────────────────────────
try:
    test_reply = groq_chat("Say 'Groq is working!' in exactly those words.")
    print("Groq test:", test_reply)
except Exception as exc:
    print("⚠ Groq API error:", exc)
    print("  Make sure GROQ_API_KEY is set correctly above.")

## ⑤ Define ZenML pipeline steps

In [ ]:
import hashlib
import json
import numpy as np
import pandas as pd
import faiss
import torch

from pathlib import Path
from datetime import datetime
from typing import Tuple, List, Dict, Any

from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

try:
    from zenml import step, pipeline
    ZENML_OK = True
except ImportError:
    # Graceful fallback: define pass-through decorators
    def step(func=None, **kwargs):
        return func if func else lambda f: f
    def pipeline(func=None, **kwargs):
        return func if func else lambda f: f
    ZENML_OK = False
    print("Running without ZenML decorator (standalone mode)")

ROOT = Path(WORK_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 1 — Ingest
# ═══════════════════════════════════════════════════════════════════
@step
def ingest_data(dataset_path: str = DATASET_PATH) -> pd.DataFrame:
    """Load and basic-clean the EduFeed Excel dataset."""
    df = pd.read_excel(dataset_path, engine="openpyxl")
    required = ["professor_name", "comments", "star_rating"]
    df = df.dropna(subset=required)
    df = df.drop_duplicates(subset=["professor_name", "comments"])
    df = df.reset_index(drop=True)
    df.insert(0, "review_id", df.index.astype(str).str.zfill(6))
    df["comments"] = df["comments"].astype(str).str.strip()
    df["star_rating"] = pd.to_numeric(df["star_rating"], errors="coerce")
    df = df[df["comments"].str.len() > 5]   # drop blank/trivial comments
    df = df.reset_index(drop=True)
    print(f"[Layer 1] Loaded {len(df):,} reviews from {df['professor_name'].nunique():,} professors")
    return df

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 2 — Anonymise
# ═══════════════════════════════════════════════════════════════════
@step
def anonymise(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """SHA-256 tokenise professor names; return anonymised df + mapping."""
    def sha256_token(name: str) -> str:
        digest = hashlib.sha256(name.lower().strip().encode()).hexdigest()[:8].upper()
        return f"PROF_{digest}"

    prof_map: Dict[str, str] = {
        n: sha256_token(n)
        for n in df["professor_name"].dropna().unique()
    }
    df = df.copy()
    df["professor_token"] = df["professor_name"].map(prof_map)
    df = df.drop(columns=["professor_name"])
    print(f"[Layer 2] Anonymised {len(prof_map):,} professors")
    return df, prof_map

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 3a — SBERT Embeddings
# ═══════════════════════════════════════════════════════════════════
@step
def embed_reviews(df: pd.DataFrame) -> np.ndarray:
    """Encode all comments with SentenceTransformer (MiniLM-L6-v2)."""
    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device=DEVICE
    )
    texts = df["comments"].tolist()
    batch = 256 if DEVICE == "cuda" else 64
    emb = model.encode(
        texts,
        batch_size=batch,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    out_path = ROOT / "layer3_embeddings.npy"
    np.save(out_path, emb)
    print(f"[Layer 3a] Embeddings shape: {emb.shape}, saved → {out_path}")
    return emb

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 3b — DeBERTa Zero-shot Sentiment
# ═══════════════════════════════════════════════════════════════════
@step
def classify_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Zero-shot sentiment via DeBERTa-v3-base NLI model.
    Labels: positive | negative | neutral
    """
    device_id = 0 if DEVICE == "cuda" else -1
    clf = hf_pipeline(
        "zero-shot-classification",
        model="cross-encoder/nli-deberta-v3-base",
        device=device_id,
        batch_size=64 if DEVICE == "cuda" else 16,
    )
    LABELS = ["positive", "negative", "neutral"]
    texts = df["comments"].tolist()
    sentiments, scores = [], []

    for i in range(0, len(texts), 64):
        batch = texts[i : i + 64]
        results = clf(batch, LABELS)
        for r in results:
            sentiments.append(r["labels"][0])
            scores.append(round(r["scores"][0], 4))
        if i % 512 == 0:
            print(f"  sentiment progress: {i}/{len(texts)}")

    df = df.copy()
    df["sentiment"]       = sentiments
    df["sentiment_score"] = scores

    out_path = ROOT / "layer3_enriched.csv"
    df.to_csv(out_path, index=False)
    print(f"[Layer 3b] Sentiment dist: {df['sentiment'].value_counts().to_dict()}")
    print(f"           Saved → {out_path}")
    return df

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 4 — FAISS Index
# ═══════════════════════════════════════════════════════════════════
@step
def build_faiss_index(emb: np.ndarray) -> faiss.swigfaiss.Index:
    """Build HNSW FAISS index over sentence embeddings."""
    dim = emb.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.add(emb.astype("float32"))
    out_path = ROOT / "layer4_faiss.index"
    faiss.write_index(index, str(out_path))
    print(f"[Layer 4] FAISS index: {index.ntotal:,} vectors (dim={dim}), saved → {out_path}")
    return index

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 5 — Groq LLM Insight Cards
# ═══════════════════════════════════════════════════════════════════
@step
def generate_groq_insights(
    df: pd.DataFrame,
    max_profs: int = MAX_PROFS,
) -> List[Dict[str, Any]]:
    """
    For each professor, build a structured insight card using:
      • Aggregated stats (rating, sentiment, tags)
      • Top-5 verbatim reviews fed to Groq LLM for natural-language summary
    """
    SYSTEM_PROMPT = (
        "You are an educational analytics assistant. "
        "Given professor reviews, provide a concise 2-3 sentence summary, "
        "3 key strengths, and 2 areas for improvement. "
        "Be factual, constructive, and professional."
    )

    prof_col   = "professor_token"
    professors = df[prof_col].value_counts().head(max_profs).index.tolist()
    cards: List[Dict[str, Any]] = []

    for prof in professors:
        prof_df = df[df[prof_col] == prof]
        reviews = prof_df["comments"].head(5).tolist()

        dept       = (prof_df["department_name"].mode().iloc[0]
                      if "department_name" in prof_df.columns else "Unknown")
        avg_rating = round(float(prof_df["star_rating"].mean()), 2)
        sentiments = prof_df["sentiment"].value_counts().to_dict()
        tags_raw   = prof_df["tag_professor"].dropna().head(3).tolist() if "tag_professor" in prof_df.columns else []

        # Build Groq prompt
        reviews_block = "\n".join(f"  {i+1}. {r[:200]}" for i, r in enumerate(reviews))
        user_prompt = (
            f"Professor: {prof}\n"
            f"Department: {dept}\n"
            f"Average Rating: {avg_rating}/5\n"
            f"Review count: {len(prof_df)}\n"
            f"Sentiment breakdown: {sentiments}\n"
            f"Top reviews:\n{reviews_block}\n\n"
            f"Provide a summary, 3 key strengths, and 2 areas for improvement."
        )

        try:
            llm_response = groq_chat(
                prompt=user_prompt,
                system=SYSTEM_PROMPT,
                max_tokens=400,
                temperature=0.3,
            )
        except Exception as exc:
            llm_response = f"[Groq error: {exc}]"

        card: Dict[str, Any] = {
            "professor":      prof,
            "department":     dept,
            "avg_rating":     avg_rating,
            "review_count":   int(len(prof_df)),
            "sentiments":     sentiments,
            "tags":           tags_raw,
            "llm_summary":    llm_response,        # ← Groq output
            "top_reviews":    reviews,
            "generated_at":   datetime.utcnow().isoformat(),
        }
        cards.append(card)
        print(f"  [Groq ✔] {prof} | {dept} | {avg_rating}★")

    out_path = ROOT / "outputs" / "insights.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(cards, f, indent=2, default=str)
    print(f"[Layer 5] {len(cards)} insight cards saved → {out_path}")
    return cards

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LAYER 6 — Export & QA
# ═══════════════════════════════════════════════════════════════════
@step
def export_and_qa(
    df: pd.DataFrame,
    emb: np.ndarray,
    cards: List[Dict[str, Any]],
) -> str:
    """Save final predictions CSV, RLHF stub, and Markdown report."""
    out_dir = ROOT / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Predictions CSV
    pred_path = out_dir / "predictions.csv"
    df.to_csv(pred_path, index=False)

    # RLHF feedback stub
    (ROOT / "rlhf_feedback.jsonl").touch()

    # Markdown report
    avg_star = round(float(df["star_rating"].mean()), 3) if "star_rating" in df.columns else "N/A"
    report_lines = [
        "# EduFeed ZenML + Groq Pipeline Report",
        f"Generated : {datetime.utcnow().isoformat()}",
        "",
        "## Stats",
        f"- Total reviews      : {len(df):,}",
        f"- Professors         : {df['professor_token'].nunique():,}",
        f"- Insight cards      : {len(cards)}",
        f"- Embedding dims     : {emb.shape[1]}",
        f"- Avg star rating    : {avg_star}",
        f"- Sentiment dist     : {df['sentiment'].value_counts().to_dict()}",
        "",
        "## LLM",
        f"- Provider  : Groq ({GROQ_BASE_URL})",
        f"- Model     : {GROQ_MODEL}",
        "",
        "## Status",
        "PIPELINE COMPLETE ✔",
    ]
    report_text = "\n".join(report_lines)
    (out_dir / "report.md").write_text(report_text)

    print("[Layer 6] Exported:")
    print(f"  predictions.csv  ({len(df):,} rows)")
    print(f"  insights.json    ({len(cards)} cards)")
    print(f"  report.md")
    print(report_text)
    return "COMPLETE"

## ⑥ Define and run the full ZenML pipeline

In [ ]:
@pipeline(name="edufeed_deberta_groq_pipeline")
def edufeed_pipeline():
    """Full 6-layer EduFeed pipeline: DeBERTa sentiment + Groq insight cards."""
    # Layer 1 — Ingest
    df_raw = ingest_data(dataset_path=DATASET_PATH)

    # Layer 2 — Anonymise
    df_anon, prof_map = anonymise(df=df_raw)

    # Layer 3a — SBERT embeddings
    embeddings = embed_reviews(df=df_anon)

    # Layer 3b — DeBERTa zero-shot sentiment
    df_enriched = classify_sentiment(df=df_anon)

    # Layer 4 — FAISS index
    faiss_index = build_faiss_index(emb=embeddings)

    # Layer 5 — Groq LLM insight cards
    insight_cards = generate_groq_insights(df=df_enriched, max_profs=MAX_PROFS)

    # Layer 6 — Export & QA
    status = export_and_qa(df=df_enriched, emb=embeddings, cards=insight_cards)

    return status


print("Pipeline defined ✔")

In [ ]:
# ── Run the pipeline ──────────────────────────────────────────────
if ZENML_OK:
    # ZenML managed run
    run = edufeed_pipeline()
    print("ZenML run complete:", run)
else:
    # Standalone run (no ZenML server needed)
    print("Running in standalone mode (ZenML decorators are pass-throughs)...")
    df_raw      = ingest_data(dataset_path=DATASET_PATH)
    df_anon, pm = anonymise(df=df_raw)
    embeddings  = embed_reviews(df=df_anon)
    df_enriched = classify_sentiment(df=df_anon)
    faiss_index = build_faiss_index(emb=embeddings)
    cards       = generate_groq_insights(df=df_enriched, max_profs=MAX_PROFS)
    status      = export_and_qa(df=df_enriched, emb=embeddings, cards=cards)
    print("Pipeline status:", status)

## ⑦ Evaluation

In [ ]:
import numpy as np
import pandas as pd
import faiss
import json
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

OUT = Path(WORK_DIR) / "outputs"

df  = pd.read_csv(OUT / "predictions.csv")
emb = np.load(Path(WORK_DIR) / "layer3_embeddings.npy")
idx = faiss.read_index(str(Path(WORK_DIR) / "layer4_faiss.index"))

with open(OUT / "insights.json") as f:
    cards = json.load(f)

# ── 1. Sentiment accuracy (ground truth available) ───────────────
print("=" * 60)
print("1. SENTIMENT ACCURACY")
print("=" * 60)

if "sentiment_label" in df.columns:
    label_map = {
        "positive": "positive", "negative": "negative", "neutral": "neutral",
        "Positive": "positive", "Negative": "negative", "Neutral": "neutral",
        1: "positive", 0: "neutral", -1: "negative",
    }
    df["gt"] = df["sentiment_label"].map(label_map).fillna("neutral")
    acc = accuracy_score(df["gt"], df["sentiment"])
    print(f"Accuracy  : {acc:.3f}  ({acc*100:.1f}%)")
    print(classification_report(df["gt"], df["sentiment"]))
    print("Confusion matrix (positive / neutral / negative):")
    print(confusion_matrix(df["gt"], df["sentiment"],
                           labels=["positive", "neutral", "negative"]))
else:
    print("No ground-truth sentiment_label column — showing prediction dist:")
    print(df["sentiment"].value_counts())

# ── 2. Retrieval quality (FAISS Hit@K) ───────────────────────────
print("\n" + "=" * 60)
print("2. RETRIEVAL QUALITY  (Hit@1 / Hit@5)")
print("=" * 60)

prof_col   = "professor_token"
sample_n   = min(200, len(df))
sample_idx = np.random.choice(len(df), sample_n, replace=False)
hit1 = hit5 = 0

for i in sample_idx:
    q = emb[i].reshape(1, -1).astype("float32")
    _, I = idx.search(q, k=6)
    neighbors = [j for j in I[0] if j != i][:5]
    true_prof = df.iloc[i][prof_col]
    nbr_profs = df.iloc[neighbors][prof_col].tolist()
    if nbr_profs[0] == true_prof: hit1 += 1
    if true_prof in nbr_profs:    hit5 += 1

print(f"Hit@1 : {hit1/sample_n:.3f}  ({hit1/sample_n*100:.1f}%)")
print(f"Hit@5 : {hit5/sample_n:.3f}  ({hit5/sample_n*100:.1f}%)")

# ── 3. Groq insight card preview ────────────────────────────────
print("\n" + "=" * 60)
print("3. GROQ INSIGHT CARDS (first 3)")
print("=" * 60)

for card in cards[:3]:
    print(f"\n▶ {card['professor']} | {card['department']} | {card['avg_rating']}★")
    print(f"  Sentiments : {card['sentiments']}")
    print(f"  LLM Summary:\n{card['llm_summary'][:400]}")

# ── 4. Overall summary ───────────────────────────────────────────
print("\n" + "=" * 60)
print("4. PIPELINE SUMMARY")
print("=" * 60)
print(f"Total reviews     : {len(df):,}")
print(f"Professors        : {df[prof_col].nunique():,}")
print(f"Insight cards     : {len(cards)}")
print(f"Embedding dims    : {emb.shape[1]}")
print(f"Avg star rating   : {df['star_rating'].mean():.3f}")
print(f"Sentiment dist    : {df['sentiment'].value_counts().to_dict()}")
print(f"LLM provider      : Groq  ({GROQ_BASE_URL})")
print(f"LLM model         : {GROQ_MODEL}")
if 'acc' in dir():
    print(f"Sentiment accuracy: {acc*100:.1f}%")
print("=" * 60)